<a href="https://colab.research.google.com/github/SJH-official/JH-Archive/blob/main/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_0417_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sqlite3와 Pydantic

- SQLite3에서 가져온 데이터는 보통 tuple이나 dict 형태인데, 이를 Pydantic 모델로 변환하면 코드의 가독성과 안정성이 비약적으로 상승

In [1]:
import sqlite3
from pydantic import BaseModel, ConfigDict
from typing import Optional

# 1. Pydantic 모델 정의 (데이터 스키마)
class User(BaseModel):
    id: int
    name: str
    email: str
    age: Optional[int] = None

    # SQLite의 Row 객체를 dict처럼 다루기 위한 설정 (Pydantic v2 기준)
    model_config = ConfigDict(from_attributes=True)

# 2. 데이터베이스 초기화 및 샘플 데이터 삽입 (강의용 세팅)
def init_db():
    conn = sqlite3.connect(":memory:")  # 테스트를 위해 메모리 DB 사용
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            email TEXT,
            age INTEGER
        )
    """)
    cursor.execute("INSERT INTO users VALUES (1, 'Alice', 'alice@example.com', 25)")
    cursor.execute("INSERT INTO users VALUES (2, 'Bob', 'bob@example.com', NULL)")
    conn.commit()
    return conn

# 3. 데이터 조회 및 Pydantic 변환 함수
def get_users(conn):
    # 결과를 dict 형태로 가져오기 위해 row_factory 설정
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users")

    rows = cursor.fetchall()

    # DB 행(Row) 데이터를 Pydantic 객체 리스트로 변환
    return [User.model_validate(dict(row)) for row in rows]

# --- 실행부 ---
connection = init_db()
user_list = get_users(connection)

for user in user_list:
    print(f"ID: {user.id} | 이름: {user.name} | 이메일: {user.email}")
    # Pydantic을 쓰면 user['name'] 대신 user.name (도트 연산자) 사용 가능!

ID: 1 | 이름: Alice | 이메일: alice@example.com
ID: 2 | 이름: Bob | 이메일: bob@example.com


## 실습과제

- https://github.com/ancestor9/data/blob/main/customers.csv 을 읽어와서 pydantic 에 저장하라

In [2]:
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the Pydantic model for customer data
class Customer(BaseModel):
    customer_id: str = Field(alias='고객ID')
    gender: str = Field(alias='성별')
    payment_method: str = Field(alias='결제수단')
    residence: str = Field(alias='거주지')
    membership_level: str = Field(alias='회원등급')
    satisfaction: int = Field(alias='만족도') # Corrected type from str to int
    recent_access_time_hour: int = Field(alias='최근접속시간(시)')
    preferred_product_temperature: float = Field(alias='선호제품군_적정온도')
    age: int = Field(alias='나이')
    purchase_quantity: int = Field(alias='구매수량')
    total_purchase_amount: int = Field(alias='총결제금액')

# URL of the CSV file
csv_url = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"

try:
    # Read the CSV into a pandas DataFrame
    df = pd.read_csv(csv_url)
    print(f"CSV file loaded successfully. Number of records: {len(df)}")
    print("\nDataFrame columns:", df.columns.tolist()) # Added to inspect columns

    # Convert DataFrame records to Pydantic model instances
    # Using .to_dict('records') to get a list of dictionaries, then validating each with Pydantic
    customer_pydantic_list: List[Customer] = [Customer.model_validate(record) for record in df.to_dict('records')]

    print("\nFirst 5 customer Pydantic objects:")
    for i, customer in enumerate(customer_pydantic_list[:5]):
        print(customer.model_dump_json(indent=2))

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the CSV URL is correct and the Pydantic model matches the CSV columns.")


CSV file loaded successfully. Number of records: 10000

DataFrame columns: ['고객ID', '성별', '결제수단', '거주지', '회원등급', '만족도', '최근접속시간(시)', '선호제품군_적정온도', '나이', '구매수량', '총결제금액']

First 5 customer Pydantic objects:
{
  "customer_id": "CUST_0001",
  "gender": "남성",
  "payment_method": "휴대폰결제",
  "residence": "대구",
  "membership_level": "Gold",
  "satisfaction": 2,
  "recent_access_time_hour": 1,
  "preferred_product_temperature": 27.0,
  "age": 34,
  "purchase_quantity": 7,
  "total_purchase_amount": 760355
}
{
  "customer_id": "CUST_0002",
  "gender": "여성",
  "payment_method": "계좌이체",
  "residence": "대구",
  "membership_level": "Gold",
  "satisfaction": 5,
  "recent_access_time_hour": 15,
  "preferred_product_temperature": 24.6,
  "age": 20,
  "purchase_quantity": 42,
  "total_purchase_amount": 727001
}
{
  "customer_id": "CUST_0003",
  "gender": "여성",
  "payment_method": "신용카드",
  "residence": "광주",
  "membership_level": "Gold",
  "satisfaction": 1,
  "recent_access_time_hour": 6,
  "preferred_

In [3]:
!pip install pydantic gradio pandas

In [ ]:
# 1. 필수 라이브러리 설치
!pip install pydantic gradio pandas

import pandas as pd
import gradio as gr
from pydantic import BaseModel, Field, ValidationError
from typing import List
import io

# 2. Pydantic 모델 정의
class Customer(BaseModel):
    id: str = Field(alias="고객ID")
    gender: str = Field(alias="성별")
    payment_method: str = Field(alias="결제수단")
    location: str = Field(alias="거주지")
    membership: str = Field(alias="회원등급")
    satisfaction: int = Field(alias="만족도", ge=1, le=5)
    age: int = Field(alias="나이", gt=0)
    total_spent: int = Field(alias="총결제금액", ge=0)

    class Config:
        populate_by_name = True

# 데이터 저장소
customer_db: List[Customer] = []

# --- [기능 함수들] ---

def load_csv(file):
    global customer_db
    if file is None: return "파일을 올려주세요."
    try:
        df = pd.read_csv(file.name)
        customer_db = [Customer(**row) for row in df.to_dict('records')]
        return f"✅ {len(customer_db)}명의 데이터를 로드했습니다."
    except Exception as e:
        return f"❌ 로드 실패: {e}"

def update_or_add_customer(c_id, gender, pay, loc, member, sat, age, spent):
    global customer_db
    try:
        new_data = Customer(고객ID=c_id, 성별=gender, 결제수단=pay, 거주지=loc,
                            회원등급=member, 만족도=sat, 나이=age, 총결제금액=spent)
        idx = next((i for i, c in enumerate(customer_db) if c.id == c_id), None)
        if idx is not None:
            customer_db[idx] = new_data
            msg = f"📝 {c_id} 고객 정보를 수정했습니다."
        else:
            customer_db.append(new_data)
            msg = f"✨ 신규 고객 {c_id}를 추가했습니다."
        return msg
    except ValidationError as e:
        return f"⚠️ 입력 오류: {e.errors()[0]['msg']}"

def search_customer(c_id):
    customer = next((c for c in customer_db if c.id == c_id), None)
    if customer:
        return pd.DataFrame([customer.dict(by_alias=True)]), f"🔍 {c_id} 고객 조회 완료"
    return None, "❌ 해당 ID의 고객이 없습니다."

def delete_customer(c_id):
    global customer_db
    initial_len = len(customer_db)
    customer_db = [c for c in customer_db if c.id != c_id]
    if len(customer_db) < initial_len:
        return f"🗑️ {c_id} 고객 데이터가 삭제되었습니다."
    return "❌ 삭제할 고객을 찾을 수 없습니다."

def get_all_data():
    if not customer_db: return None
    return pd.DataFrame([c.dict(by_alias=True) for c in customer_db])

def export_csv():
    if not customer_db: return None
    df = pd.DataFrame([c.dict(by_alias=True) for c in customer_db])
    path = "updated_customers.csv"
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

# --- [Gradio UI] ---

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏢 고객 관리 시스템 (Gradio Live)")

    with gr.Tab("📂 데이터 로드"):
        file_input = gr.File(label="CSV 업로드")
        load_btn = gr.Button("데이터 불러오기", variant="primary")
        status_msg = gr.Textbox(label="상태")
        load_btn.click(load_csv, inputs=file_input, outputs=status_msg)

    with gr.Tab("🔍 조회 및 삭제"):
        search_id = gr.Textbox(label="고객 ID 입력")
        with gr.Row():
            search_btn = gr.Button("조회")
            delete_btn = gr.Button("삭제", variant="stop")
        search_res_msg = gr.Textbox(label="결과")
        search_df = gr.DataFrame()
        search_btn.click(search_customer, inputs=search_id, outputs=[search_df, search_res_msg])
        delete_btn.click(delete_customer, inputs=search_id, outputs=search_res_msg)

    with gr.Tab("➕ 추가 및 수정"):
        with gr.Row():
            in_id = gr.Textbox(label="고객ID")
            in_gen = gr.Dropdown(["남성", "여성", "기타"], label="성별")
            in_mem = gr.Dropdown(["Bronze", "Silver", "Gold", "VIP", "VVIP"], label="회원등급")
        with gr.Row():
            in_pay = gr.Dropdown(["신용카드", "계좌이체", "간편결제", "휴대폰결제"], label="결제수단")
            in_loc = gr.Textbox(label="거주지")
            in_sat = gr.Slider(1, 5, step=1, label="만족도")
        with gr.Row():
            in_age = gr.Number(label="나이")
            in_spent = gr.Number(label="총결제금액")
        save_btn = gr.Button("저장하기", variant="primary")
        save_res = gr.Textbox(label="결과")
        save_btn.click(update_or_add_customer,
                       inputs=[in_id, in_gen, in_pay, in_loc, in_mem, in_sat, in_age, in_spent],
                       outputs=save_res)

    with gr.Tab("📊 현황 및 다운로드"):
        with gr.Row():
            refresh_btn = gr.Button("전체 새로고침")
            download_btn = gr.Button("CSV로 저장", variant="secondary")
        full_df = gr.DataFrame()
        file_out = gr.File(label="파일 다운로드")
        refresh_btn.click(get_all_data, outputs=full_df)
        download_btn.click(export_csv, outputs=file_out)

# share=True를 추가하여 외부 라이브 링크 생성
demo.launch(share=True, debug=True)

/tmp/ipykernel_7538/3773527772.py:11: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Customer(BaseModel):
/tmp/ipykernel_7538/3773527772.py:82: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb4aed703c643b5cf6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_7538/3773527772.py:71: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return pd.DataFrame([c.dict(by_alias=True) for c in customer_db])
